# CNN for Image Classification (multi-class classification)

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

**torchvision** -> its is a official Pytorch library spacilly for computer vision
- its have **datasets** - CIFAR10, MNIST
- its also have lot of - **pretrained CNNS**
- and some importent tools or **utilities** for image transform  

## DAtasets & DataLoader

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # its help to perform transformation on images

# image => scale (0, 1) => normalize => (-1, 1)
transform = transforms.Compose([
    # auto convert all images to pytorch tensor and auto scale all images
    transforms.ToTensor(), # Scale [0, 255] => [0.0, 1.0]
    
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # in this params(mean, std)  define - what std-dev and mean values do we need after normalizing the image? 
])

# training set and testing set are automaticly available in CIFAR10, only we get
trainset = CIFAR10(root="./data", train=True, download=True, transform=transform)
testset = CIFAR10(root="./data", train=False, download=True, transform=transform)

100%|██████████| 170M/170M [00:03<00:00, 45.5MB/s] 


#### **transforms.`compose`**: 
- its help to chain the lot of image transformation
- we don't apply on this time
- only we set all transformation earlier for the apply every image.
- its help us to do multi-transformation

#### **transforms.ToTensor()**:
- It automatically divides all pixel values by **255.0** to scale them from the range **[0, 255] down to [0.0, 1.0]**.

#### **transforms.Normailize(mean, std):**
- PyTorch uses these inputs to apply the standard normalization formula to every pixel:
    $\frac{Input - mean}{std}$

#### (0.5, 0.5, 0.5) yields the [-1, 1] range
passing a **mean** of `0.5` and a standard deviation of `0.5` shifts teh data mathematically:
- for the minimum value (0):
$\frac{0-0.5}{0.5}$ = $\frac{-0.5}{0.5}$ = -1.0
- for the maximum value (1):
$\frac{1-0.5}{0.5}$ = $\frac{0.5}{0.5}$ = 1.0

two sets of three values `(0.5, 0.5, 0.5)` map directly to RGB channels of image

In [3]:
trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
testloader = DataLoader(testset, batch_size=64)

## Build CNN
- we get (32, 32, 3) -> (32, 32, 32)

- we use **3 convolutional layers** - mixed of convo + ReLU and maxpool
- convolutional layer is 2D beacuse we deal with images

- CNN => **[convolution+ReLU -> maxpool]**-3time => **flattening** => **FCL**
- convol - {kernel = 3 x 3 , padding = 1, stride = 1 (default)}
- maxpooling - {kernel = 2 x 2 , stride = 2}

- In **FCL** -> 1 HL (Linear + ReLU Activation func.)

-  than: **Final Output LAyer** - Linear Activation Func. (auto softmax beacuse of CrossEntropyLoss)

In [16]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        # 1st layer of CNN : Feature Extractors
        # each Image size = 32x32
        self.conv_layers = nn.Sequential(
            # 1st convolution layer
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1),
            nn.ReLU(), # convo+ReLU -> (32, 32, 3) => (32, 32, 32)
            nn.MaxPool2d(kernel_size=2, stride=2), # (16, 16, 32)
                
            # 2nd convolution layer
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
            nn.ReLU(), # convo+ReLU -> (16, 16, 32) => (16, 16, 64)
            nn.MaxPool2d(kernel_size=2, stride=2), # (8, 8, 64)
            
            # 3rd convolution layer
            nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, padding=1),
            nn.ReLU(), # convo+ReLU => (8, 8, 64) => (8, 8, 128)
            nn.MaxPool2d(kernel_size=2, stride=2) # (4, 4, 128)
            # after flatten the (4, 4, 128) => 4*48*128 inputs
        )

        # 2nd layer of CNN
        self.fc_layers  = nn.Sequential(
            nn.Linear(4*4*128, 256), # input, output=random
            nn.ReLU(),

            nn.Linear(256, 10) # o/p=10 because of multi-class classification
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # flattening | by standord method
        x = self.fc_layers(x)

        return x

#### **conv2D()** : 2d => for image  
-  its jobs is to **slide small filters** across the image to detect basic visual features like edges, lines, textures, and shapes. 

- **in_channels** : how many color layers the input image has.
- **out_channels** : represents the number of filters (or feature detectors) this layer will learn
- **kernel_size=3** : sets the dimensions of the sliding filter windowm to 3x3 pixels.
- **stride**=1: by default **s=1**

> The layer looks at a tiny 3x3 patch (kernal_size=3) of the image at a time, calculates a mathematical value based on what it sees, and then moves over by 1 pixel (stride=1) to look at the next patch.

- **padding=1** : adds a 1-pixel thick border of zeros all the way around the outside of image

> This allows the filter to scan the outermost edge pixels perfectly, ensuring that the output image maintains the exact same width and height (WxH) it had before entering this layer.

#### **maxPool2d()** : down sampling 
- when we apply **maxPool2d(2, 2)** it Downsample dimensions by half

**After Three Pooling**  
- A 32x32 image becomes 4x4 : `(32 -> 16 -> 8 -> 4)`  

**final Output image**  
- final **produced images** after all 3 convolution layers is => **(4, 4, 128)**
> 128 channels * 4 * 4 pixels = `128*4*4` = _ fatures

In [17]:
model = CNN()
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [21]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss funx
        loss.backward() # BF
        optimizer.step() # update params

        epoch_training_loss += loss.item()
    
    print(f"epoch={epoch+1} & loss={epoch_training_loss/len(trainloader)}")

epoch=1 & loss=1.007846692562713
epoch=2 & loss=0.7800146935464781
epoch=3 & loss=0.6422261340005319
epoch=4 & loss=0.5251959795918306
epoch=5 & loss=0.4245767926278017
epoch=6 & loss=0.3346245591921727
epoch=7 & loss=0.25755290090180266
epoch=8 & loss=0.1950094377279015
epoch=9 & loss=0.1500529206531775
epoch=10 & loss=0.12176217030986305


In [22]:
# Evaluation 

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        output = model(images)
        
        _, predicted = torch.max(output, 1)

        correct_labels += (predicted==labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels / total_labels * 100}")



Accuracy = 76.27000000000001
